# Notebook 05: Encoder-Decoder Architecture

## Why This Matters

Encoder-decoder transformers underlie some of the most commercially important AI systems: translation APIs (Google Translate, DeepL), document summarization pipelines, speech recognition systems (Whisper), and code generation with structured output. While decoder-only LLMs have dominated headlines, encoder-decoder models remain the preferred architecture when you have long structured inputs and need precise, bounded outputs — a machine translation model that must render a full sentence faithfully is a different problem than a chatbot. T5 and its successors showed that "text-to-text" unification can work, and understanding the cross-attention mechanism is fundamental to understanding how the decoder extracts structured information from an encoded representation.

## Background

The original "Attention Is All You Need" transformer is an encoder-decoder model. The encoder reads the full source sequence (e.g., a French sentence) and produces contextualized representations. The decoder, conditioned on those representations via cross-attention, generates the target sequence (e.g., English translation) one token at a time.

**Three major model families:**
- **Encoder-only** (BERT, RoBERTa): bidirectional understanding; best for classification, NER, retrieval
- **Decoder-only** (GPT, LLaMA): autoregressive generation; best for open-ended generation, instruction following
- **Encoder-decoder** (T5, BART, Whisper): best for seq2seq with long input and structured output

**T5 (Text-to-Text Transfer Transformer):** Reframed all NLP as text-to-text: classification becomes "classify: [input] → positive/negative", translation is "translate en to fr: [text] → [translation]". This unified framing enables multitask pre-training.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Cross-Attention: The Bridge Between Encoder and Decoder

Cross-attention is where the decoder "reads" the encoder's output. The key distinction from self-attention:

- **Queries (Q):** come from the **decoder**'s current hidden state  
- **Keys (K) and Values (V):** come from the **encoder**'s output

$$\text{CrossAttention}(Q_{\text{dec}}, K_{\text{enc}}, V_{\text{enc}}) = \text{softmax}\left(\frac{Q_{\text{dec}} K_{\text{enc}}^T}{\sqrt{d_k}}\right) V_{\text{enc}}$$

**Intuition:** Each decoder position asks "what encoder context is relevant to generating my next token?" The encoder positions with high attention weight are the "evidence" the decoder uses. For translation, attention patterns often show the decoder attending to the corresponding source word when generating a target word.

**KV caching for cross-attention:** Unlike self-attention, where the KV cache grows with generated tokens, the cross-attention K, V are fixed for the entire generation (they come from the encoder, which only runs once). This makes cross-attention essentially free during decoding — the encoder runs once, and then the cross-attention is just a matrix multiply with the pre-computed encoder representations.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return self.W_o(attn_out), attn_weights

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.gelu(self.linear1(x))))

print("Core modules loaded.")

## 2. Full Encoder-Decoder Transformer

The decoder layer has three sublayers (vs two in the encoder):
1. **Causal self-attention** on the decoder sequence (with causal mask)
2. **Cross-attention** — queries from decoder, keys/values from encoder
3. **Feed-forward network**

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, mask=src_mask)
        x = residual + self.dropout(attn_out)
        residual = x
        x = residual + self.dropout(self.ffn(self.norm2(x)))
        return x

class TransformerDecoderLayer(nn.Module):
    """Full encoder-decoder decoder layer: causal self-attn + cross-attn + FFN."""
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, memory_mask=None):
        # 1. Causal self-attention on decoder sequence
        residual = x
        x_norm = self.norm1(x)
        self_attn_out, _ = self.self_attn(x_norm, x_norm, x_norm, mask=tgt_mask)
        x = residual + self.dropout(self_attn_out)

        # 2. Cross-attention: decoder queries attend to encoder keys/values
        residual = x
        x_norm = self.norm2(x)
        cross_attn_out, cross_weights = self.cross_attn(
            x_norm,         # queries from decoder
            enc_output,     # keys from encoder
            enc_output,     # values from encoder
            mask=memory_mask
        )
        x = residual + self.dropout(cross_attn_out)

        # 3. Feed-forward
        residual = x
        x = residual + self.dropout(self.ffn(self.norm3(x)))
        return x, cross_weights

class EncoderDecoderTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads,
                 n_enc_layers, n_dec_layers, d_ff=None, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        # Embeddings
        self.src_embed = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model)
        self.src_pos = nn.Embedding(max_len, d_model)
        self.tgt_pos = nn.Embedding(max_len, d_model)
        # Encoder
        self.enc_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_enc_layers)
        ])
        # Decoder
        self.dec_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_dec_layers)
        ])
        self.enc_norm = nn.LayerNorm(d_model)
        self.dec_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, tgt_vocab_size, bias=False)
        self.dropout = nn.Dropout(dropout)

    def encode(self, src, src_mask=None):
        B, T = src.shape
        pos = torch.arange(T, device=src.device).unsqueeze(0)
        x = self.dropout(self.src_embed(src) + self.src_pos(pos))
        for layer in self.enc_layers:
            x = layer(x, src_mask)
        return self.enc_norm(x)

    def decode(self, tgt, enc_output, tgt_mask=None, memory_mask=None):
        B, T = tgt.shape
        pos = torch.arange(T, device=tgt.device).unsqueeze(0)
        x = self.dropout(self.tgt_embed(tgt) + self.tgt_pos(pos))
        cross_weights_all = []
        for layer in self.dec_layers:
            x, cw = layer(x, enc_output, tgt_mask, memory_mask)
            cross_weights_all.append(cw)
        return self.dec_norm(x), cross_weights_all

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encode(src, src_mask)
        dec_output, cross_weights = self.decode(tgt, enc_output, tgt_mask)
        return self.lm_head(dec_output), cross_weights

# Build and test
def make_causal_mask(T, device):
    return torch.triu(torch.ones(T, T, device=device), diagonal=1).bool().unsqueeze(0).unsqueeze(0)

src_vocab, tgt_vocab, d_model, n_heads = 500, 600, 128, 8
model = EncoderDecoderTransformer(src_vocab, tgt_vocab, d_model, n_heads,
                                   n_enc_layers=3, n_dec_layers=3).to(device)
src = torch.randint(0, src_vocab, (2, 15)).to(device)
tgt = torch.randint(0, tgt_vocab, (2, 10)).to(device)
tgt_mask = make_causal_mask(10, device)
logits, cross_weights = model(src, tgt, tgt_mask=tgt_mask)
print(f"Encoder-Decoder logits: {logits.shape}")       # (2, 10, 600)
print(f"Cross-attn weights layers: {len(cross_weights)}")  # 3 layers
print(f"Cross-attn weight shape:  {cross_weights[0].shape}")  # (2, 8, 10, 15)

## 3. Teacher Forcing vs Autoregressive Inference

**Teacher forcing (training):** At every position in the decoder, we feed the ground-truth previous token, not the model's predicted token. This allows the full target sequence to be processed in parallel:

- Input to decoder: `[<BOS>, token_1, token_2, ..., token_{T-1}]`  
- Target: `[token_1, token_2, ..., token_T, <EOS>]`
- Loss: cross-entropy averaged over all positions

**Why teacher forcing?** Without it, errors compound (exposure bias): if the model generates a wrong token early, subsequent generations condition on that wrong token, making training unstable. Teacher forcing breaks this feedback loop during training.

**Exposure bias problem:** The model is trained on ground-truth prefixes but sees its own generated tokens at inference — this distributional shift can cause degraded performance. Scheduled sampling (Bengio et al. 2015) gradually replaces ground-truth tokens with model predictions during training to close this gap.

In [ ]:
def make_translation_batch(src_vocab, tgt_vocab, B=4, src_len=12, tgt_len=8):
    """Simulate a training batch for sequence-to-sequence."""
    src = torch.randint(1, src_vocab, (B, src_len))
    # tgt_in: [BOS] + tokens (teacher forcing input)
    tgt_in = torch.randint(1, tgt_vocab, (B, tgt_len))
    tgt_in[:, 0] = 0  # BOS token = 0
    # tgt_out: tokens + [EOS] (target for loss)
    tgt_out = torch.cat([tgt_in[:, 1:], torch.zeros(B, 1, dtype=torch.long)], dim=1)
    return src, tgt_in, tgt_out

src, tgt_in, tgt_out = make_translation_batch(src_vocab, tgt_vocab)
print(f"Source:          {src.shape}")
print(f"Decoder input:   {tgt_in.shape}")
print(f"Decoder target:  {tgt_out.shape}")

# Training step
model_cpu = EncoderDecoderTransformer(src_vocab, tgt_vocab, d_model, n_heads,
                                       n_enc_layers=2, n_dec_layers=2)
optimizer = torch.optim.AdamW(model_cpu.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# One training step
tgt_mask = make_causal_mask(tgt_in.size(1), 'cpu')
logits, _ = model_cpu(src, tgt_in, tgt_mask=tgt_mask)
# Reshape for CrossEntropy: (B*T, vocab) vs (B*T,)
loss = criterion(logits.view(-1, tgt_vocab), tgt_out.view(-1))
loss.backward()
optimizer.step()
optimizer.zero_grad()
print(f"Training step loss: {loss.item():.4f}")

## 4. Autoregressive Decoding and Beam Search

**Greedy decoding:** At each step, pick the highest-probability token. Fast but suboptimal — the globally best sequence might require a locally lower-probability choice.

**Beam search:** Maintain a beam of $k$ candidate sequences. At each step, expand each candidate by all vocabulary items, keep the top-$k$ by cumulative log-probability. This explores more of the search space without being exhaustive.

$$\text{score}(y_1, \ldots, y_T) = \sum_{t=1}^{T} \log P(y_t | y_{<t}, x)$$

**Beam search length normalization:** Longer sequences have lower cumulative log-prob. To prevent beam search from favoring short outputs, scores are normalized by length:

$$\text{score}_{\text{norm}} = \frac{1}{T^{\alpha}} \sum_{t=1}^{T} \log P(y_t | y_{<t}, x), \quad \alpha \in [0.6, 0.8]$$

**When beam search helps vs hurts:**
- **Translation:** Large improvements over greedy (2-3 BLEU points for beam=4)
- **Open-ended generation:** Beam search produces repetitive, generic text ("mode collapse" to the center of the distribution). Nucleus/top-p sampling is preferred.
- **Code generation:** Beam search is competitive for short function generation

In [ ]:
@torch.no_grad()
def beam_search_decode(model, src, bos_id=0, eos_id=1, beam_size=4,
                        max_len=30, length_alpha=0.6):
    """
    Simple beam search for encoder-decoder transformer.
    Returns list of (score, token_ids) tuples, sorted by score.
    """
    model.eval()
    vocab_size = model.lm_head.out_features

    # Encode source (once)
    enc_out = model.encode(src)  # (1, src_len, d_model)

    # Initialize beam: each element is (log_prob, token_list)
    beams = [(0.0, [bos_id])]
    completed = []

    for step in range(max_len):
        if not beams:
            break

        candidates = []
        for log_prob, tokens in beams:
            if tokens[-1] == eos_id:
                completed.append((log_prob / (len(tokens) ** length_alpha), tokens))
                continue

            tgt = torch.tensor([tokens], dtype=torch.long)
            tgt_mask = make_causal_mask(len(tokens), 'cpu')
            dec_out, _ = model.decode(tgt, enc_out, tgt_mask=tgt_mask)
            # Get next token distribution
            log_probs = F.log_softmax(model.lm_head(dec_out[:, -1, :]), dim=-1)

            # Expand: take top-k next tokens
            top_log_probs, top_ids = log_probs[0].topk(beam_size)
            for lp, tid in zip(top_log_probs.tolist(), top_ids.tolist()):
                candidates.append((log_prob + lp, tokens + [tid]))

        # Keep top beam_size candidates
        beams = sorted(candidates, key=lambda x: -x[0] / (len(x[1]) ** length_alpha))[:beam_size]

    # Add any incomplete beams
    for log_prob, tokens in beams:
        completed.append((log_prob / (len(tokens) ** length_alpha), tokens))

    return sorted(completed, reverse=True)[:beam_size]

model_cpu = EncoderDecoderTransformer(200, 200, 64, 4, n_enc_layers=2, n_dec_layers=2)
src_ex = torch.randint(2, 200, (1, 8))
results = beam_search_decode(model_cpu, src_ex, bos_id=0, eos_id=1, beam_size=3, max_len=10)
print("Beam search results (random model):")
for score, tokens in results:
    print(f"  score={score:.3f}, tokens={tokens}")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Encoder-decoder | Encoder: full bidirectional context; Decoder: causal self-attn + cross-attn |
| Cross-attention | Decoder Q attends to encoder K, V; bridges source and target |
| Cross-attn KV cache | Encoder runs once; K,V fixed for entire generation (free during decoding) |
| Teacher forcing | Feed ground-truth previous tokens during training; enables parallel training |
| Exposure bias | Train on gold, test on predictions; scheduled sampling partially bridges gap |
| Beam search | Maintain $k$ candidates; better than greedy for translation/summarization |
| Length normalization | Divide cumulative log-prob by $T^\alpha$; prevents short-sequence bias |
| T5 text-to-text | Unify all NLP as text-to-text; enables multitask pre-training on T5 |
| Encoder-decoder vs dec-only | Enc-dec: best for long-input/bounded-output; dec-only: best for open generation |